# 01 EDA: CCPD2019 sample
Runs on Colab (Drive mounted). Data-prep logic lives in `src/alpr/ccpd.py`; plots and tables live here.
CCPD license: Zenodo says CC BY 4.0, original repo terms unchecked, so this notebook shows statistics and plots only, no sample images.

In [ ]:
# Parameters
SEED = 0
DRIVE = "/content/drive/MyDrive/alpr"
ARCHIVES = f"{DRIVE}/archives"
ARCHIVE = f"{ARCHIVES}/CCPD2019.tar.xz"
MANIFEST = f"{ARCHIVES}/ccpd_manifest.csv"
SAMPLE_TAR = f"{ARCHIVES}/ccpd_sample.tar"

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

In [ ]:
import os, shutil
from alpr import ccpd  # ImportError on the first run? Restart the kernel once: an editable install is not visible to an already-running kernel.

os.makedirs(ARCHIVES, exist_ok=True)
print("alpr.ccpd:", ccpd.__file__)
print("local disk free GB:", round(shutil.disk_usage("/content").free / 1e9, 1))
print("Drive free GB:", round(shutil.disk_usage("/content/drive").free / 1e9, 1))
print("archive on Drive:", os.path.exists(ARCHIVE))

## Step 4: data prep
Each cell is safe to rerun: download resumes from the partial file and verifies MD5, sample and build skip if their output exists (build writes to `.part` first). Keep the Colab session connected while they run.

In [ ]:
# 4a. Download CCPD2019 (13.2 GB) straight to Drive and verify MD5
!python -u -m alpr.ccpd download "{ARCHIVE}"

In [ ]:
# 4b. Pass 1: list the archive, check the subfolders, draw the stratified sample (seed above)
!python -u -m alpr.ccpd sample "{ARCHIVE}" "{MANIFEST}" --seed {SEED}

In [ ]:
# 4c. Pass 2: stream the archive again, copy only the sampled images into one .tar (+ manifest.csv) on Drive
!python -u -m alpr.ccpd build "{ARCHIVE}" "{MANIFEST}" "{SAMPLE_TAR}"

In [ ]:
# 4d. Verify the packed sample against the manifest
import collections, tarfile
rows = ccpd.read_manifest(MANIFEST)
with tarfile.open(SAMPLE_TAR) as tf:
    names = [n for n in tf.getnames() if n != "manifest.csv"]
print("manifest rows:", len(rows), "| images in tar:", len(names))
print("tar per split:", dict(collections.Counter(n.split("/")[0] for n in names)))
assert sorted(n.split("/", 1)[1] for n in names) == sorted(m.rsplit("/", 1)[1] for m, _, _ in rows)
print("sample tar:", round(os.path.getsize(SAMPLE_TAR) / 1e9, 2), "GB, MD5", ccpd.file_md5(SAMPLE_TAR))